# 05 — Model Evaluation

**Project:** IT22638168 — MaternaLink FER track
**Pipeline position:** step 5 of 8 · see `../README.md`

## Purpose

- Evaluate on the held-out test set. Touch it ONCE.
- Report accuracy AND macro-F1 (accuracy alone misleads on imbalanced data).
- Per-class precision, recall, F1.
- Confusion matrix -> plots/.
- CHECKPOINT CP-1: inspect calm/neutral confusion specifically.
- Confidence distribution — informs tau_face_min.

## Before running

Read `../../../docs/system/MOOD_STATE_SPEC.md` — it defines the label space this model targets.

> **Note:** CP-1: if calm and neutral are not separable, revise MOOD_STATE_SPEC.md to two substantive states + unknown via a decision memo. That is an acceptable outcome.

## Status

Not yet run.


---

## Work starts here

## 0. Environment setup &mdash; repo root, DATA_ROOT (WSL2), artifact dirs

Same portability logic as notebooks 03/04: `find_repo_root()` never depends on CWD, and
`DATA_ROOT` prefers the WSL-native copy (`~/fer/data/raw`) over the `/mnt/c` repo copy, with a
loud warning if the slow mount is used.

In [1]:
import os
import sys
import time

# --- repo root: walk up until ml/fer/notebooks is found ----------------------
def find_repo_root(start=None, max_up=8):
    """Auto-detect the repository root; never depends on the CWD being the notebook dir."""
    cur = os.path.abspath(start or os.getcwd())
    tried = []
    for _ in range(max_up):
        tried.append(cur)
        if os.path.isdir(os.path.join(cur, "ml", "fer", "notebooks")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise FileNotFoundError(
        "Could not locate the repository root (a directory containing ml/fer/notebooks). "
        f"Directories tried, walking up from the CWD: {tried}"
    )

REPO_ROOT = find_repo_root()
FER_ROOT = os.path.join(REPO_ROOT, "ml", "fer")
print("REPO_ROOT:", REPO_ROOT)
print("FER_ROOT: ", FER_ROOT)
print("CWD:      ", os.getcwd())
print("Platform: ", sys.platform, "|", os.uname().release if hasattr(os, "uname") else "n/a")


# --- DATA_ROOT: prefer the WSL-native copy, fall back to the repo copy --------
def find_data_root(search_root, max_depth=3):
    """Return the directory that directly contains both 'train' and 'test'."""
    if not os.path.isdir(search_root):
        raise FileNotFoundError(f"search root does not exist: {search_root}")
    queue = [(search_root, 0)]
    visited = []
    while queue:
        current, depth = queue.pop(0)
        try:
            entries = os.listdir(current)
        except OSError:
            continue
        visited.append(current)
        lower = {e.lower(): e for e in entries}
        if "train" in lower and "test" in lower:
            tr = os.path.join(current, lower["train"])
            te = os.path.join(current, lower["test"])
            if os.path.isdir(tr) and os.path.isdir(te):
                return current
        if depth < max_depth:
            for e in entries:
                sub = os.path.join(current, e)
                if os.path.isdir(sub):
                    queue.append((sub, depth + 1))
    raise FileNotFoundError(
        f"no directory containing both 'train' and 'test' within {max_depth} levels of "
        f"{search_root}. Examined: {visited}"
    )

DATA_ROOT_CANDIDATES = [
    ("wsl-native", os.path.expanduser("~/fer/data/raw")),
    ("repo-mnt-c", os.path.join(FER_ROOT, "data", "raw")),
]

DATA_ROOT = None
DATA_ROOT_SOURCE = None
print()
print("Resolving DATA_ROOT (preference order: WSL-native, then the repo copy on /mnt/c):")
for _label, _cand in DATA_ROOT_CANDIDATES:
    print(f"  [{_label}] {_cand} -> exists={os.path.isdir(_cand)}")
    if DATA_ROOT is None and os.path.isdir(_cand):
        try:
            DATA_ROOT = find_data_root(_cand)
            DATA_ROOT_SOURCE = _label
        except FileNotFoundError as _e:
            print(f"      rejected: {_e}")

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Neither candidate data root is usable. Copy the FER-2013 raw tree into the WSL native "
        "filesystem first:  mkdir -p ~/fer/data && cp -r "
        f"{os.path.join(FER_ROOT, 'data', 'raw')} ~/fer/data/"
    )

print()
print("Resolved DATA_ROOT:", DATA_ROOT, f"(source: {DATA_ROOT_SOURCE})")

SLOW_MOUNT = DATA_ROOT.startswith("/mnt/")
if SLOW_MOUNT:
    print()
    print("!" * 78)
    print("!!  WARNING - reading images from the WINDOWS MOUNT (/mnt/c).                !!")
    print("!!                                                                          !!")
    print("!!  Measured on this machine (see notebook 04): 800 JPEGs took 17.0 s from   !!")
    print("!!  /mnt/c versus 2.7 s from the WSL-native filesystem - a 6.4x penalty per   !!")
    print("!!  file, paid through the 9p/drvfs translation layer.                       !!")
    print("!!                                                                          !!")
    print("!!  This is a single test-set inference pass (3589 images), so the penalty   !!")
    print("!!  is paid once - but it is still avoidable. Fix it with:                   !!")
    print("!!      mkdir -p ~/fer/data && cp -r <repo>/ml/fer/data/raw ~/fer/data/      !!")
    print("!" * 78)
else:
    print("Reading images from the WSL-native filesystem - no /mnt/c I/O penalty.")

# --- artifact directories: straight into the repo, no zip/download step ------
OUT_DIR    = os.path.join(FER_ROOT, "outputs")
PLOT_DIR   = os.path.join(FER_ROOT, "plots")
NB05_PLOT_DIR = os.path.join(PLOT_DIR, "nb05")
MODELS_DIR = os.path.join(FER_ROOT, "models")
for _d in (OUT_DIR, PLOT_DIR, NB05_PLOT_DIR, MODELS_DIR):
    os.makedirs(_d, exist_ok=True)
print()
print("OUT_DIR:      ", OUT_DIR)
print("PLOT_DIR:     ", PLOT_DIR)
print("NB05_PLOT_DIR:", NB05_PLOT_DIR)
print("MODELS_DIR:   ", MODELS_DIR)

# --- byte-budget tracker (same discipline as notebooks 02/03/04) -------------
_WRITTEN_FILES = []

def track_write(path, bucket="artifact"):
    """Record a file this notebook wrote, for the end-of-notebook byte report."""
    size = os.path.getsize(path)
    _WRITTEN_FILES.append((path, size))
    return size

def bucket_bytes(files):
    return int(sum(sz for _, sz in files))

REPO_ROOT: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168
FER_ROOT:  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer
CWD:       /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/notebooks
Platform:  linux | 6.6.87.2-microsoft-standard-WSL2

Resolving DATA_ROOT (preference order: WSL-native, then the repo copy on /mnt/c):
  [wsl-native] /home/yasinduslpredetor/fer/data/raw -> exists=True
  [repo-mnt-c] /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/data/raw -> exists=True

Resolved DATA_ROOT: /home/yasinduslpredetor/fer/data/raw (source: wsl-native)
Reading images from the WSL-native filesystem - no /mnt/c I/O penalty.

OUT_DIR:       /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs
PLOT_DIR:      /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots
NB05_PLOT_DIR: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb05
MODELS_DIR:    /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer

In [2]:
import tensorflow as tf

print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__)

GPUS = tf.config.list_physical_devices("GPU")
print("tf.config.list_physical_devices('GPU'):", GPUS)

# Inference-only notebook: a GPU speeds up the single forward pass over 3589
# images x 2 models, but unlike notebook 04 (7 full trainings) it is not a hard
# requirement, so no RuntimeError is raised for a CPU-only kernel.
if GPUS:
    for _gpu in GPUS:
        try:
            tf.config.experimental.set_memory_growth(_gpu, True)
            print(f"  set_memory_growth(True) on {_gpu.name}")
        except RuntimeError as _e:
            print(f"  could not set memory growth on {_gpu.name}: {_e}")
    GPU_DEVICE_NAME = GPUS[0].name
    try:
        _details = tf.config.experimental.get_device_details(GPUS[0])
        GPU_DEVICE_DESCRIPTION = _details.get("device_name", "unknown")
    except Exception:
        GPU_DEVICE_DESCRIPTION = "unknown"
    print("GPU device name:       ", GPU_DEVICE_NAME)
    print("GPU device description:", GPU_DEVICE_DESCRIPTION)
    ENVIRONMENT = "wsl2-gpu"
else:
    print("No GPU visible - running inference on CPU. Slower, but a single pass over the")
    print("test set is a few thousand images, not a training loop, so this is tolerable.")
    GPU_DEVICE_NAME = None
    GPU_DEVICE_DESCRIPTION = None
    ENVIRONMENT = "wsl2-cpu"

# No mixed precision here: both saved models were trained/saved with their own
# policy baked into the graph/weights; inference does not need mixed_float16
# and keeping compute in float32 keeps the evaluation numerically simple.
print()
print(f"ENVIRONMENT = {ENVIRONMENT!r}")

I0000 00:00:1787875058.312036    5226 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787875058.540761    5226 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787875060.725158    5226 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1
tf.config.list_physical_devices('GPU'): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
  set_memory_growth(True) on /physical_device:GPU:0
GPU device name:        /physical_device:GPU:0
GPU device description: NVIDIA GeForce RTX 3050 6GB Laptop GPU

ENVIRONMENT = 'wsl2-gpu'


In [3]:
"""Run metadata (redefinition) - the stub cell above is the project's fixed template;
this cell replaces its RUN/save_run with the fuller schema used by notebooks 03/04, so
this run's record has the same shape (package_versions, dataset_version as a content
hash, model_version, etc.) as the runs it will be compared against.
"""
import json, os, time, random, sys, hashlib, importlib.metadata
import numpy as np

def _pkg_version(name, module=None, alt_dist_names=None):
    """Best-effort package version lookup; never raises."""
    for dist_name in [name] + list(alt_dist_names or []):
        try:
            return importlib.metadata.version(dist_name)
        except Exception:
            continue
    try:
        mod = module or __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "unknown"

PACKAGE_VERSIONS = {
    "python":       sys.version.split()[0],
    "numpy":        _pkg_version("numpy"),
    "pandas":       _pkg_version("pandas"),
    "Pillow":       _pkg_version("Pillow", module=__import__("PIL")),
    "matplotlib":   _pkg_version("matplotlib"),
    "seaborn":      _pkg_version("seaborn"),
    "scikit-learn": _pkg_version("scikit-learn", module=__import__("sklearn")),
    "tensorflow":   _pkg_version(
        "tensorflow", module=tf,
        alt_dist_names=["tensorflow-cpu", "tensorflow-gpu", "tensorflow-intel"],
    ),
    "keras":        _pkg_version("keras", module=tf.keras),
}
print("Package versions:")
for _k, _v in PACKAGE_VERSIONS.items():
    print(f"  {_k:<13} {_v}")

# --- dataset_version: manifest filename + SHA-256 computed at runtime --------
MANIFEST_PATH = os.path.join(OUT_DIR, "splits_cleaned.csv")
if not os.path.isfile(MANIFEST_PATH):
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}. Run notebook 02 first (it writes splits_cleaned.csv)."
    )

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
DATASET_VERSION = f"{os.path.basename(MANIFEST_PATH)}@sha256:{MANIFEST_SHA256}"
print()
print("Manifest:        ", MANIFEST_PATH)
print("Manifest SHA-256:", MANIFEST_SHA256)

RUN = {
    "run_id":           time.strftime("run_%Y%m%d_%H%M%S"),
    "timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook":         "05_model_evaluation",
    "dataset":          "FER-2013",
    "dataset_path":     DATA_ROOT,
    "dataset_root_source": DATA_ROOT_SOURCE,
    "dataset_version":  DATASET_VERSION,
    "model_version":    None,      # set below once both models are loaded
    "environment":      ENVIRONMENT,
    "gpu_device":       GPU_DEVICE_NAME,
    "gpu_description":  GPU_DEVICE_DESCRIPTION,
    "package_versions": PACKAGE_VERSIONS,
    "hyperparameters":  {},        # populated at the end from the actual eval config
    "random_seed":      42,
    "metrics":          {},        # populated at the end from measured TEST-set values only
    "notes":            "",
}

random.seed(RUN["random_seed"])
np.random.seed(RUN["random_seed"])
tf.keras.utils.set_random_seed(RUN["random_seed"])

print()
print(json.dumps({k: v for k, v in RUN.items() if k not in ("package_versions",)}, indent=2))

def save_run(run=RUN, outdir=None):
    """Write the run record. Safe to call repeatedly; the last call wins."""
    if outdir is None:
        outdir = OUT_DIR
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, run["run_id"] + ".json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2)
    print("saved:", path)
    return path

RUN["run_id"]

Package versions:
  python        3.11.15
  numpy         2.4.6
  pandas        3.0.5
  Pillow        12.3.0
  matplotlib    3.11.1
  seaborn       0.13.2
  scikit-learn  1.9.0
  tensorflow    2.21.0
  keras         3.15.1

Manifest:         /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/splits_cleaned.csv
Manifest SHA-256: a75d460f35d14c14cba6830923e5e5483f6d5611e779a9252d0b57fd49325496

{
  "run_id": "run_20260828_052743",
  "timestamp": "2026-08-28 05:27:43",
  "notebook": "05_model_evaluation",
  "dataset": "FER-2013",
  "dataset_path": "/home/yasinduslpredetor/fer/data/raw",
  "dataset_root_source": "wsl-native",
  "dataset_version": "splits_cleaned.csv@sha256:a75d460f35d14c14cba6830923e5e5483f6d5611e779a9252d0b57fd49325496",
  "model_version": null,
  "environment": "wsl2-gpu",
  "gpu_device": "/physical_device:GPU:0",
  "gpu_description": "NVIDIA GeForce RTX 3050 6GB Laptop GPU",
  "hyperparameters": {},
  "random_seed": 42,
  "metrics": {},
  "notes": ""
}

'run_20260828_052743'

## 1. Load the TEST manifest &mdash; hard gate

The held-out test set (`split_group == "TEST"`, `prefix == "PrivateTest_"`) is touched exactly
once in this whole project, and this is that one place. Before any image is decoded or any model
is loaded, the manifest slice is verified against the known-correct shape: **exactly 3589 rows,
every one of them `PrivateTest_`**. This is a hard gate - it raises rather than warns, per the
supervisor's protocol, because a wrong test slice here would invalidate every number below it.

In [4]:
import pandas as pd

manifest_raw = pd.read_csv(MANIFEST_PATH)
print(f"Manifest rows read: {len(manifest_raw)}")
print("Columns:", list(manifest_raw.columns))
print()
print("split_group counts:")
print(manifest_raw["split_group"].value_counts().to_string())

test_df = manifest_raw[manifest_raw["split_group"] == "TEST"].reset_index(drop=True).copy()

# --- HARD GATE: exact row count -----------------------------------------------
N_TEST_EXPECTED = 3589
n_test_measured = len(test_df)
if n_test_measured != N_TEST_EXPECTED:
    raise AssertionError(
        f"TEST split size mismatch: expected {N_TEST_EXPECTED}, measured {n_test_measured}. "
        f"The manifest at {MANIFEST_PATH} does not match the shape this notebook was written "
        "against - STOP. The test set is touched exactly once; do not proceed on a wrong slice."
    )
print()
print(f"[PASS] TEST row count == {N_TEST_EXPECTED}")

# --- HARD GATE: every TEST row is a PrivateTest_ file -------------------------
_bad_prefix = test_df.loc[test_df["prefix"] != "PrivateTest_", "prefix"].unique().tolist()
if _bad_prefix:
    raise AssertionError(
        f"TEST split contains non-PrivateTest_ prefixes: {_bad_prefix}. This is not the held-out "
        "FER-2013 PrivateTest set - STOP."
    )
print(f"[PASS] every TEST row has prefix == 'PrivateTest_' ({test_df['prefix'].nunique()} unique value)")

# --- rebuild absolute paths, exactly as notebooks 03/04 do -------------------
def rebuild_path(file_path, split_group, cls):
    split_dir = "train" if split_group == "TRAIN" else "test"
    basename = os.path.basename(str(file_path).replace("\\", "/"))
    return os.path.join(DATA_ROOT, split_dir, str(cls), basename)

test_df["basename"] = [
    os.path.basename(str(p).replace("\\", "/")) for p in test_df["file_path"]
]
test_df["abs_path"] = [
    rebuild_path(p, g, c)
    for p, g, c in zip(test_df["file_path"], test_df["split_group"], test_df["class"])
]
print()
print("Path rebuild example:")
print("  raw     :", test_df["file_path"].iloc[0])
print("  rebuilt :", test_df["abs_path"].iloc[0])

print()
print("Verifying every rebuilt TEST path exists on disk...")
_t0 = time.time()
_missing = [p for p in test_df["abs_path"] if not os.path.isfile(p)]
print(f"  checked {len(test_df)} paths in {time.time() - _t0:.1f}s")
if _missing:
    print(f"  [FAIL] {len(_missing)} rebuilt paths do NOT exist. First 10:")
    for p in _missing[:10]:
        print("    ", p)
    raise FileNotFoundError(
        f"{len(_missing)} rebuilt TEST paths are missing from disk - DATA_ROOT is wrong or the "
        f"raw dataset copy is incomplete. DATA_ROOT={DATA_ROOT}"
    )
print("  [PASS] all rebuilt TEST paths exist.")

# --- label space: SORTED class list, identical rule to notebooks 03/04 -------
CLASS_NAMES = sorted(test_df["class"].unique().tolist())
N_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
print()
print("Class -> integer label mapping (sorted class list, same rule as notebooks 03/04):")
for c in CLASS_NAMES:
    print(f"  {CLASS_TO_IDX[c]} -> {c}")
assert N_CLASSES == 7, f"expected 7 FER-2013 classes, found {N_CLASSES}: {CLASS_NAMES}"
_EXPECTED_CLASS_ORDER = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
assert CLASS_NAMES == _EXPECTED_CLASS_ORDER, (
    f"class order does not match the expected alphabetical order used by notebooks 03/04: "
    f"{CLASS_NAMES} != {_EXPECTED_CLASS_ORDER}"
)
print("[PASS] class order matches notebooks 03/04 (angry=0 ... surprise=6).")

test_paths = test_df["abs_path"].tolist()
test_basenames = test_df["basename"].tolist()
y_test = np.array([CLASS_TO_IDX[c] for c in test_df["class"]], dtype=np.int64)

print()
print(f"test_paths: {len(test_paths)}  y_test: {y_test.shape}")
print("Per-class counts (TEST), measured:")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {c:<9} n={int((y_test == i).sum()):>5}")

def assert_test_only(*path_lists, label=""):
    """The mirror-image guard of notebooks 03/04's assert_no_test: here everything MUST be TEST."""
    bad = []
    for plist in path_lists:
        for p in plist:
            b = os.path.basename(str(p).replace("\\", "/"))
            if not b.startswith("PrivateTest_"):
                bad.append(p)
    if bad:
        raise AssertionError(
            f"NON-TEST FILE LEAKED INTO EVALUATION{' in ' + label if label else ''}: "
            f"{len(bad)} non-PrivateTest_ files. First: {bad[0]}"
        )
    return True

assert_test_only(test_paths, label="section 1 path arrays")
print()
print("[PASS] every path in test_paths is a PrivateTest_ file.")

Manifest rows read: 34079
Columns: ['file_path', 'split_group', 'class', 'prefix']

split_group counts:
split_group
TRAIN    26901
TEST      3589
VAL       3589

[PASS] TEST row count == 3589
[PASS] every TEST row has prefix == 'PrivateTest_' (1 unique value)

Path rebuild example:
  raw     : ../data/raw\test\angry\PrivateTest_10131363.jpg
  rebuilt : /home/yasinduslpredetor/fer/data/raw/test/angry/PrivateTest_10131363.jpg

Verifying every rebuilt TEST path exists on disk...
  checked 3589 paths in 0.0s
  [PASS] all rebuilt TEST paths exist.

Class -> integer label mapping (sorted class list, same rule as notebooks 03/04):
  0 -> angry
  1 -> disgust
  2 -> fear
  3 -> happy
  4 -> neutral
  5 -> sad
  6 -> surprise
[PASS] class order matches notebooks 03/04 (angry=0 ... surprise=6).

test_paths: 3589  y_test: (3589,)
Per-class counts (TEST), measured:
  angry     n=  491
  disgust   n=   55
  fear      n=  528
  happy     n=  879
  neutral   n=  626
  sad       n=  594
  surprise  n=

## 2. `tf.data` pipeline over TEST &mdash; identical preprocessing to notebooks 03/04

Same decode/preprocess recipe, unchanged: `tf.io.decode_jpeg(..., dct_method="INTEGER_ACCURATE")`
(bit-exact with PIL, so reproducible), grayscale replicated to 3 channels, bilinear resize
48x48 -> 96x96, then `mobilenet_v2.preprocess_input` (-> `[-1, 1]`). No augmentation, no shuffling
- this is evaluation, not training. Reads happen from `DATA_ROOT` (WSL-native, `/mnt/c` fallback)
resolved above.

In [5]:
AUTOTUNE = tf.data.AUTOTUNE
NATIVE_SIZE = 48          # FER-2013 native resolution; NOT a model input size
INPUT_SIZE  = 96          # model input, inherited from notebook 03's decision
EVAL_BATCH_SIZE = 64      # matches notebook 04's BATCH_SIZE; inference-only so VRAM headroom is ample

def _decode_and_preprocess(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=1, dct_method="INTEGER_ACCURATE")
    # dct_method pinned: TF's default IDCT is platform/version dependent and differs from
    # PIL by up to ~13 grey levels. INTEGER_ACCURATE is bit-exact with PIL (see notebook 04).
    img = tf.image.resize(img, (NATIVE_SIZE, NATIVE_SIZE), method="bilinear")
    img = tf.cast(tf.round(img), tf.uint8)                              # (48, 48, 1) uint8
    x = tf.cast(img, tf.float32)
    x = tf.image.grayscale_to_rgb(x)                                    # (48, 48, 3)
    x = tf.image.resize(x, (INPUT_SIZE, INPUT_SIZE), method="bilinear")  # (96, 96, 3)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)          # -> [-1, 1]
    return x

def make_eval_dataset(paths, labels, batch_size=EVAL_BATCH_SIZE):
    """Never shuffled, never augmented, never weighted - this is TEST-set inference."""
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths, dtype=tf.string), tf.constant(labels, dtype=tf.int64))
    )
    ds = ds.map(
        lambda p, y: (_decode_and_preprocess(p), y),
        num_parallel_calls=AUTOTUNE, deterministic=True,
    )
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

TEST_DS = make_eval_dataset(test_paths, y_test)

# --- shape / range verification on one batch --------------------------------
_xb, _yb = next(iter(TEST_DS))
print("TEST batch: x", _xb.shape, _xb.dtype,
      f"range [{float(tf.reduce_min(_xb)):.3f}, {float(tf.reduce_max(_xb)):.3f}]",
      " y", _yb.shape, _yb.dtype)
assert tuple(_xb.shape[1:]) == (INPUT_SIZE, INPUT_SIZE, 3), "unexpected TEST batch shape"
del _xb, _yb

# --- TEST dataset order is fixed: predictions must align with y_test row for row --
_test_labels_from_ds = np.concatenate([yb.numpy() for _, yb in TEST_DS], axis=0)
assert np.array_equal(_test_labels_from_ds, y_test), \
    "the TEST dataset re-orders rows - it must not be shuffled/deterministic=True violated"
print("[PASS] TEST dataset preserves manifest row order (labels match y_test exactly).")
del _test_labels_from_ds

print()
print(f"TEST_DS ready: {len(test_paths)} images, batch_size={EVAL_BATCH_SIZE}, "
      f"{N_CLASSES}-way softmax target.")

I0000 00:00:1787875063.411672    5226 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3617 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


TEST batch: x (64, 96, 96, 3) <dtype: 'float32'> range [-1.000, 1.000]  y (64,) <dtype: 'int64'>
[PASS] TEST dataset preserves manifest row order (labels match y_test exactly).

TEST_DS ready: 3589 images, batch_size=64, 7-way softmax target.


## 3. Load both models

`fer_mobilenetv2_finetuned_96.keras` (notebook 04, the primary/winning model) and
`fer_mobilenetv2_frozen_96.keras` (notebook 03, the frozen-backbone baseline). Both are
evaluated on the same TEST set in this notebook.

In [6]:
MODEL_PATHS = {
    "finetuned": os.path.join(MODELS_DIR, "fer_mobilenetv2_finetuned_96.keras"),
    "frozen":    os.path.join(MODELS_DIR, "fer_mobilenetv2_frozen_96.keras"),
}

for _tag, _p in MODEL_PATHS.items():
    if not os.path.isfile(_p):
        raise FileNotFoundError(
            f"model file not found for '{_tag}': {_p}. Run notebook "
            f"{'04' if _tag == 'finetuned' else '03'} first - it saves this file."
        )

MODELS = {}
MODEL_INFO = {}
for _tag, _p in MODEL_PATHS.items():
    print(f"Loading '{_tag}' model: {_p}")
    _t0 = time.time()
    _m = tf.keras.models.load_model(_p)
    _load_s = time.time() - _t0
    _size_bytes = os.path.getsize(_p)
    _total_params = int(_m.count_params())
    _trainable_params = int(sum(int(np.prod(v.shape)) for v in _m.trainable_variables))
    MODELS[_tag] = _m
    MODEL_INFO[_tag] = {
        "path": _p,
        "file_size_mb": _size_bytes / 1e6,
        "total_params": _total_params,
        "trainable_params": _trainable_params,
        "load_seconds": _load_s,
    }
    print(f"  loaded in {_load_s:.1f}s  |  {_size_bytes / 1e6:.1f} MB on disk  |  "
          f"total_params={_total_params:,}  trainable_params={_trainable_params:,}")

print()
print("Model comparison:")
_info_df = pd.DataFrame(MODEL_INFO).T
print(_info_df[["file_size_mb", "total_params", "trainable_params"]].to_string(
    formatters={"file_size_mb": "{:.1f}".format,
                "total_params": "{:,}".format,
                "trainable_params": "{:,}".format}
))

Loading 'finetuned' model: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_finetuned_96.keras
  loaded in 2.7s  |  24.5 MB on disk  |  total_params=2,266,951  trainable_params=1,848,583
Loading 'frozen' model: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_frozen_96.keras
  loaded in 1.3s  |  9.7 MB on disk  |  total_params=2,266,951  trainable_params=8,967

Model comparison:
          file_size_mb total_params trainable_params
finetuned         24.5    2,266,951        1,848,583
frozen             9.7    2,266,951            8,967


## 4. Run inference for BOTH models &mdash; one pass over TEST, no threshold

Every prediction is the raw 7-way softmax; nothing here applies `tau_face_min` or any other
confidence threshold - that decision belongs to a later notebook, not this one.

In [7]:
assert_test_only(test_paths, label="section 4 inference input")

PROBS = {}          # tag -> (N, 7) float32 softmax
PREDS = {}          # tag -> (N,) int predicted label
INFER_SECONDS = {}

for _tag, _m in MODELS.items():
    print(f"Predicting with '{_tag}' model over {len(test_paths)} TEST images ...")
    _t0 = time.time()
    _probs = _m.predict(TEST_DS, verbose=0).astype(np.float32)
    _el = time.time() - _t0
    assert _probs.shape == (len(test_paths), N_CLASSES), \
        f"'{_tag}' produced probs shape {_probs.shape}, expected {(len(test_paths), N_CLASSES)}"
    _rowsums = _probs.sum(axis=1)
    assert np.allclose(_rowsums, 1.0, atol=1e-3), f"'{_tag}' softmax rows do not sum to 1"
    _preds = _probs.argmax(axis=1)
    PROBS[_tag] = _probs
    PREDS[_tag] = _preds
    INFER_SECONDS[_tag] = _el
    print(f"  done in {_el:.1f}s ({len(test_paths) / max(_el, 1e-9):.0f} img/s), "
          f"softmax rows verified to sum to 1, no threshold applied.")

# --- alignment: prediction order must equal test_df/y_test row order ---------
for _tag in MODELS:
    assert len(PREDS[_tag]) == len(y_test), f"'{_tag}' prediction count mismatch"
print()
print("[PASS] both models' predictions are aligned row-for-row with y_test / test_basenames.")

Predicting with 'finetuned' model over 3589 TEST images ...


I0000 00:00:1787875068.759281    5329 service.cc:153] XLA service 0x7b62b805a950 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787875068.759338    5329 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6 (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1787875068.820093    5329 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787875069.305156    5329 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1787875079.111642    5329 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  done in 18.5s (194 img/s), softmax rows verified to sum to 1, no threshold applied.
Predicting with 'frozen' model over 3589 TEST images ...
  done in 6.0s (596 img/s), softmax rows verified to sum to 1, no threshold applied.

[PASS] both models' predictions are aligned row-for-row with y_test / test_basenames.


## 5. Metrics &mdash; accuracy, macro-F1, weighted-F1, per-class precision/recall/F1

Accuracy alone misleads on FER-2013's class imbalance (disgust has ~10x fewer samples than
happy), so macro-F1 (unweighted mean over classes) is reported alongside it, plus the full
per-class breakdown for both models.

In [8]:
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)

REPORTS = {}          # tag -> dict[class] -> {precision, recall, f1, support} (+ 'accuracy' key raw)
HEADLINE = {}          # tag -> {accuracy, macro_f1, weighted_f1}

for _tag in MODELS:
    _preds = PREDS[_tag]
    _acc = float(accuracy_score(y_test, _preds))
    _macro_f1 = float(f1_score(y_test, _preds, average="macro"))
    _weighted_f1 = float(f1_score(y_test, _preds, average="weighted"))
    _report = classification_report(
        y_test, _preds, labels=list(range(N_CLASSES)), target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    REPORTS[_tag] = _report
    HEADLINE[_tag] = {"accuracy": _acc, "macro_f1": _macro_f1, "weighted_f1": _weighted_f1}

print("=" * 78)
print("CANONICAL TEST-SET METRICS (full 3589-row TEST set, no exclusions, no threshold)")
print("=" * 78)
_headline_df = pd.DataFrame(HEADLINE).T
print(_headline_df.to_string(formatters={c: "{:.4f}".format for c in _headline_df.columns}))

for _tag in MODELS:
    print()
    print(f"--- '{_tag}' per-class report ---")
    print(classification_report(
        y_test, PREDS[_tag], labels=list(range(N_CLASSES)), target_names=CLASS_NAMES,
        zero_division=0, digits=4,
    ))

# --- majority-class / random-guess sanity floors, computed at runtime --------
_majority_idx = int(np.bincount(y_test, minlength=N_CLASSES).argmax())
_majority_rate = float((y_test == _majority_idx).sum() / len(y_test))
_macro_f1_floor = 1.0 / N_CLASSES
print()
print(f"Majority class on TEST: '{CLASS_NAMES[_majority_idx]}' (rate={_majority_rate:.4f})")
print(f"Random-guess macro-F1 floor: {_macro_f1_floor:.4f}")
for _tag in MODELS:
    _beats_majority = HEADLINE[_tag]["accuracy"] > _majority_rate
    _beats_floor = HEADLINE[_tag]["macro_f1"] > _macro_f1_floor
    print(f"  [{'PASS' if _beats_majority else 'FAIL'}] '{_tag}' accuracy beats majority-class rate")
    print(f"  [{'PASS' if _beats_floor else 'FAIL'}] '{_tag}' macro-F1 beats the random-guess floor")

CANONICAL TEST-SET METRICS (full 3589-row TEST set, no exclusions, no threshold)
          accuracy macro_f1 weighted_f1
finetuned   0.6289   0.6036      0.6262
frozen      0.4795   0.4138      0.4704

--- 'finetuned' per-class report ---
              precision    recall  f1-score   support

       angry     0.5533    0.4969    0.5236       491
     disgust     0.6750    0.4909    0.5684        55
        fear     0.4701    0.4773    0.4737       528
       happy     0.8143    0.8783    0.8451       879
     neutral     0.5714    0.6262    0.5976       626
         sad     0.4982    0.4697    0.4835       594
    surprise     0.7698    0.6995    0.7330       416

    accuracy                         0.6289      3589
   macro avg     0.6218    0.5913    0.6036      3589
weighted avg     0.6260    0.6289    0.6262      3589


--- 'frozen' per-class report ---
              precision    recall  f1-score   support

       angry     0.4038    0.2994    0.3439       491
     disgust     0.2

## 6. Confusion matrices &mdash; CHECKPOINT CP-1 (calm/neutral confusion)

Raw-count and row-normalised confusion matrices for both models, saved as heatmaps (no face
images anywhere - axis labels are class names only). The top 5 largest off-diagonal cells per
model are also printed explicitly, which is where CP-1 (calm/neutral separability) gets
inspected: FER-2013 has no "calm" class, so this is read as fear/sad/neutral cross-confusion.

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

CM_RAW = {}
CM_NORM = {}

for _tag in MODELS:
    _cm = confusion_matrix(y_test, PREDS[_tag], labels=list(range(N_CLASSES)))
    _cm_norm = _cm.astype(np.float64) / _cm.sum(axis=1, keepdims=True)
    CM_RAW[_tag] = _cm
    CM_NORM[_tag] = _cm_norm

def plot_confusion_matrix(cm, class_names, title, path, normalised=False):
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    fmt = ".2f" if normalised else "d"
    sns.heatmap(
        cm, annot=True, fmt=fmt, cmap="Blues", cbar=True,
        xticklabels=class_names, yticklabels=class_names,
        vmin=0.0, vmax=(1.0 if normalised else None), ax=ax,
    )
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    track_write(path)
    print("saved:", path)

plot_confusion_matrix(
    CM_RAW["finetuned"], CLASS_NAMES,
    "Fine-tuned MobileNetV2 - TEST confusion matrix (raw counts)",
    os.path.join(NB05_PLOT_DIR, "confusion_matrix_finetuned_raw.png"), normalised=False,
)
plot_confusion_matrix(
    CM_NORM["finetuned"], CLASS_NAMES,
    "Fine-tuned MobileNetV2 - TEST confusion matrix (row-normalised)",
    os.path.join(NB05_PLOT_DIR, "confusion_matrix_finetuned_normalised.png"), normalised=True,
)
plot_confusion_matrix(
    CM_NORM["frozen"], CLASS_NAMES,
    "Frozen-backbone MobileNetV2 - TEST confusion matrix (row-normalised)",
    os.path.join(NB05_PLOT_DIR, "confusion_matrix_frozen_normalised.png"), normalised=True,
)

# --- top-5 largest off-diagonal cells per model, printed as text -------------
def top_off_diagonal(cm, class_names, k=5):
    n = cm.shape[0]
    total = cm.sum()
    cells = [
        (i, j, int(cm[i, j])) for i in range(n) for j in range(n) if i != j and cm[i, j] > 0
    ]
    cells.sort(key=lambda t: t[2], reverse=True)
    rows = []
    for i, j, count in cells[:k]:
        pct = 100.0 * count / total
        rows.append((class_names[i], class_names[j], count, pct))
    return rows

for _tag in MODELS:
    print()
    print(f"Top 5 largest off-diagonal confusion cells - '{_tag}' model (of {CM_RAW[_tag].sum()} TEST samples):")
    for true_c, pred_c, count, pct in top_off_diagonal(CM_RAW[_tag], CLASS_NAMES, k=5):
        print(f"  {true_c} -> {pred_c}: {count} samples ({pct:.2f}%)")

saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb05/confusion_matrix_finetuned_raw.png
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb05/confusion_matrix_finetuned_normalised.png
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb05/confusion_matrix_frozen_normalised.png

Top 5 largest off-diagonal confusion cells - 'finetuned' model (of 3589 TEST samples):
  sad -> neutral: 115 samples (3.20%)
  fear -> sad: 94 samples (2.62%)
  sad -> fear: 92 samples (2.56%)
  neutral -> sad: 77 samples (2.15%)
  angry -> fear: 71 samples (1.98%)

Top 5 largest off-diagonal confusion cells - 'frozen' model (of 3589 TEST samples):
  sad -> neutral: 156 samples (4.35%)
  fear -> sad: 128 samples (3.57%)
  angry -> neutral: 110 samples (3.06%)
  happy -> neutral: 109 samples (3.04%)
  fear -> neutral: 97 samples (2.70%)


## 7. Generalisation gap &mdash; TEST macro-F1 vs each model's VALIDATION baseline

Baselines are **read programmatically** from the run-metadata JSON files already on disk, never
hard-coded:

- **Frozen model** (notebook 03): `run_20260822_175944.json` &mdash; the later of the two
  notebook-03 runs, the one that carries `seed_variance`/`noise_floor_macro_f1`
  (`run_20260822_171451.json` is the earlier, pre-seed-repeat run). Its top-level
  `metrics.val_macro_f1` / `metrics.val_accuracy` are the actual saved-model numbers (the
  `res96_unweighted` seed-42 run), which is what corresponds to the `.keras` file loaded above.
- **Fine-tuned model** (notebook 04): `run_20260823_163030.json` &mdash; same logic, top-level
  `metrics.val_macro_f1`/`val_accuracy` are the saved winner's (seed 42, DEEP/bn_frozen/unweighted)
  own validation numbers, not the 3-seed mean.

<!-- nb03 reported macro-F1 ~0.4357 (mean over 3 seeds, frozen); nb04 reported ~0.6063 (mean over
     3 seeds, fine-tuned) - these are commented reference only; the gap computation below uses
     the JSON-loaded, model-specific values, not these numbers. -->

In [10]:
BASELINE_JSON_PATHS = {
    "frozen":    os.path.join(OUT_DIR, "run_20260822_175944.json"),
    "finetuned": os.path.join(OUT_DIR, "run_20260823_163030.json"),
}

VAL_BASELINE = {}
for _tag, _p in BASELINE_JSON_PATHS.items():
    if not os.path.isfile(_p):
        raise FileNotFoundError(
            f"baseline run-metadata JSON not found for '{_tag}': {_p}. Re-run notebook "
            f"{'03' if _tag == 'frozen' else '04'} or check the filename."
        )
    with open(_p, "r", encoding="utf-8") as f:
        _run = json.load(f)
    assert _run.get("model_version") == os.path.basename(MODEL_PATHS[_tag]), (
        f"baseline JSON {_p} model_version={_run.get('model_version')!r} does not match the "
        f"loaded model {os.path.basename(MODEL_PATHS[_tag])!r}"
    )
    _m = _run["metrics"]
    VAL_BASELINE[_tag] = {
        "source_run_id": _run["run_id"],
        "source_path": _p,
        "val_accuracy": float(_m["val_accuracy"]),
        "val_macro_f1": float(_m["val_macro_f1"]),
        "per_class": {
            c: {"precision": float(_m["per_class"][c]["precision"]),
                "recall": float(_m["per_class"][c]["recall"]),
                "f1": float(_m["per_class"][c]["f1"])}
            for c in CLASS_NAMES
        },
    }
    print(f"'{_tag}' VALIDATION baseline loaded from {_p}  "
          f"(run_id={_run['run_id']}): val_accuracy={VAL_BASELINE[_tag]['val_accuracy']:.4f}  "
          f"val_macro_f1={VAL_BASELINE[_tag]['val_macro_f1']:.4f}")

# --- generalisation gap: TEST - VAL, both metrics, both models ---------------
GAPS = {}
print()
print("=" * 78)
print("GENERALISATION GAP  (TEST macro-F1 / accuracy  minus  VALIDATION baseline)")
print("=" * 78)
for _tag in MODELS:
    _gap_macro_f1 = HEADLINE[_tag]["macro_f1"] - VAL_BASELINE[_tag]["val_macro_f1"]
    _gap_acc = HEADLINE[_tag]["accuracy"] - VAL_BASELINE[_tag]["val_accuracy"]
    GAPS[_tag] = {"macro_f1_gap": _gap_macro_f1, "accuracy_gap": _gap_acc}
    print(f"'{_tag}':")
    print(f"  VAL  macro_f1={VAL_BASELINE[_tag]['val_macro_f1']:.4f}  "
          f"accuracy={VAL_BASELINE[_tag]['val_accuracy']:.4f}")
    print(f"  TEST macro_f1={HEADLINE[_tag]['macro_f1']:.4f}  "
          f"accuracy={HEADLINE[_tag]['accuracy']:.4f}")
    print(f"  GAP  macro_f1={_gap_macro_f1:+.4f}  accuracy={_gap_acc:+.4f}  "
          f"({'test worse than val' if _gap_macro_f1 < 0 else 'test at/above val'})")

# --- plot: val vs test macro-F1, grouped bars, both models -------------------
fig, ax = plt.subplots(figsize=(7, 5))
_x = np.arange(len(MODELS))
_width = 0.35
_tags = list(MODELS.keys())
_val_vals = [VAL_BASELINE[t]["val_macro_f1"] for t in _tags]
_test_vals = [HEADLINE[t]["macro_f1"] for t in _tags]
ax.bar(_x - _width / 2, _val_vals, _width, label="VAL (baseline)")
ax.bar(_x + _width / 2, _test_vals, _width, label="TEST (this notebook)")
ax.set_xticks(_x)
ax.set_xticklabels(_tags)
ax.set_ylabel("macro-F1")
ax.set_ylim(0, 1)
ax.axhline(1.0 / N_CLASSES, color="grey", linestyle=":", linewidth=1.2,
           label=f"random-guess macro-F1 (1/{N_CLASSES})")
ax.set_title("Validation vs TEST macro-F1, both models")
ax.legend()
for i, (v, t) in enumerate(zip(_val_vals, _tags)):
    ax.text(i - _width / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
for i, (v, t) in enumerate(zip(_test_vals, _tags)):
    ax.text(i + _width / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
fig.tight_layout()
_p1 = os.path.join(NB05_PLOT_DIR, "test_vs_val_summary.png")
fig.savefig(_p1, dpi=150)
plt.close(fig)
track_write(_p1)
print()
print("saved:", _p1)

# --- plot: per-class F1, test vs val, fine-tuned model (primary) -------------
fig, ax = plt.subplots(figsize=(9, 5))
_x = np.arange(N_CLASSES)
_val_f1 = [VAL_BASELINE["finetuned"]["per_class"][c]["f1"] for c in CLASS_NAMES]
_test_f1 = [REPORTS["finetuned"][c]["f1-score"] for c in CLASS_NAMES]
ax.bar(_x - _width / 2, _val_f1, _width, label="VAL F1")
ax.bar(_x + _width / 2, _test_f1, _width, label="TEST F1")
ax.set_xticks(_x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_ylabel("F1 score")
ax.set_ylim(0, 1)
ax.set_title("Fine-tuned model - per-class F1, TEST vs VALIDATION")
ax.legend()
fig.tight_layout()
_p2 = os.path.join(NB05_PLOT_DIR, "per_class_f1_test_vs_val.png")
fig.savefig(_p2, dpi=150)
plt.close(fig)
track_write(_p2)
print("saved:", _p2)

'frozen' VALIDATION baseline loaded from /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/run_20260822_175944.json  (run_id=run_20260822_175944): val_accuracy=0.4778  val_macro_f1=0.4403
'finetuned' VALIDATION baseline loaded from /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/run_20260823_163030.json  (run_id=run_20260823_163030): val_accuracy=0.6319  val_macro_f1=0.6121

GENERALISATION GAP  (TEST macro-F1 / accuracy  minus  VALIDATION baseline)
'finetuned':
  VAL  macro_f1=0.6121  accuracy=0.6319
  TEST macro_f1=0.6036  accuracy=0.6289
  GAP  macro_f1=-0.0085  accuracy=-0.0031  (test worse than val)
'frozen':
  VAL  macro_f1=0.4403  accuracy=0.4778
  TEST macro_f1=0.4138  accuracy=0.4795
  GAP  macro_f1=-0.0265  accuracy=+0.0017  (test worse than val)

saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb05/test_vs_val_summary.png
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb05/per_class_f

## 8. Sensitivity analysis &mdash; excluding quality-anomaly images (SECONDARY table)

`image_flags.csv` (notebook 02's BlazeFace/duplicate/corruption audit) flags a subset of TEST
images as `is_anomaly == True`. This section recomputes accuracy and macro-F1 for both models
with those images excluded, joined on `file_path`. **The canonical, full-TEST numbers in section 5
remain the headline result** - this table is secondary, reported to show whether the flagged
images are dragging the numbers down materially or not.

In [11]:
FLAGS_PATH = os.path.join(OUT_DIR, "image_flags.csv")
if not os.path.isfile(FLAGS_PATH):
    raise FileNotFoundError(f"image_flags.csv not found: {FLAGS_PATH}. Run notebook 02 first.")

flags_df = pd.read_csv(FLAGS_PATH)
flags_test = flags_df[flags_df["split_group"] == "TEST"].copy()
print(f"image_flags.csv TEST rows: {len(flags_test)}")

# --- join on file_path (identical raw string format to splits_cleaned.csv) ---
_anomaly_paths = set(
    flags_test.loc[flags_test["is_anomaly"].astype(bool), "file_path"].astype(str)
)
print(f"TEST images flagged is_anomaly == True: {len(_anomaly_paths)}")

test_df["is_anomaly"] = test_df["file_path"].astype(str).isin(_anomaly_paths)
_n_anomaly_matched = int(test_df["is_anomaly"].sum())
print(f"TEST rows matched to an anomaly flag by file_path join: {_n_anomaly_matched}")
assert _n_anomaly_matched == len(_anomaly_paths), (
    f"join mismatch: {len(_anomaly_paths)} anomaly file_paths in image_flags.csv but only "
    f"{_n_anomaly_matched} matched in test_df - file_path formatting may differ between the "
    "two CSVs."
)

_keep_mask = ~test_df["is_anomaly"].to_numpy()
_n_excluded = int((~_keep_mask).sum())
_n_kept = int(_keep_mask.sum())
print()
print(f"Excluding {_n_excluded} quality-anomaly TEST images -> {_n_kept} / {len(test_df)} remain "
      f"for the sensitivity-filtered metrics below.")

SENSITIVITY = {}
for _tag in MODELS:
    _y_f = y_test[_keep_mask]
    _preds_f = PREDS[_tag][_keep_mask]
    _acc_f = float(accuracy_score(_y_f, _preds_f))
    _macro_f1_f = float(f1_score(_y_f, _preds_f, average="macro"))
    SENSITIVITY[_tag] = {"accuracy": _acc_f, "macro_f1": _macro_f1_f, "n_excluded": _n_excluded,
                          "n_kept": _n_kept}

print()
print("=" * 78)
print(f"SECONDARY TABLE - sensitivity to {_n_excluded} excluded quality-anomaly images")
print("(canonical, full-TEST numbers from section 5 remain the headline result)")
print("=" * 78)
_sens_rows = []
for _tag in MODELS:
    _sens_rows.append({
        "model": _tag,
        "canonical_accuracy": HEADLINE[_tag]["accuracy"],
        "filtered_accuracy": SENSITIVITY[_tag]["accuracy"],
        "canonical_macro_f1": HEADLINE[_tag]["macro_f1"],
        "filtered_macro_f1": SENSITIVITY[_tag]["macro_f1"],
    })
_sens_df = pd.DataFrame(_sens_rows).set_index("model")
_sens_df["accuracy_delta"] = _sens_df["filtered_accuracy"] - _sens_df["canonical_accuracy"]
_sens_df["macro_f1_delta"] = _sens_df["filtered_macro_f1"] - _sens_df["canonical_macro_f1"]
print(_sens_df.to_string(formatters={c: "{:+.4f}".format if "delta" in c else "{:.4f}".format
                                      for c in _sens_df.columns}))

image_flags.csv TEST rows: 3589
TEST images flagged is_anomaly == True: 8
TEST rows matched to an anomaly flag by file_path join: 8

Excluding 8 quality-anomaly TEST images -> 3581 / 3589 remain for the sensitivity-filtered metrics below.

SECONDARY TABLE - sensitivity to 8 excluded quality-anomaly images
(canonical, full-TEST numbers from section 5 remain the headline result)
          canonical_accuracy filtered_accuracy canonical_macro_f1 filtered_macro_f1 accuracy_delta macro_f1_delta
model                                                                                                            
finetuned             0.6289            0.6297             0.6036            0.6043        +0.0008        +0.0008
frozen                0.4795            0.4803             0.4138            0.4146        +0.0008        +0.0008


## 9. Confidence distribution (fine-tuned model) &mdash; informs `tau_face_min`

Histogram of the max-softmax probability across all TEST predictions from the fine-tuned model.
**No threshold line is drawn** - choosing `tau_face_min` is a downstream decision, not this
notebook's.

In [12]:
_max_conf = PROBS["finetuned"].max(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(_max_conf, bins=40, range=(0, 1), color="#4C72B0", edgecolor="white")
ax.set_xlabel("max softmax probability (fine-tuned model, TEST predictions)")
ax.set_ylabel("count")
ax.set_title("Confidence distribution - fine-tuned model, TEST set (no threshold applied)")
fig.tight_layout()
_p_conf = os.path.join(NB05_PLOT_DIR, "confidence_distribution_test.png")
fig.savefig(_p_conf, dpi=150)
plt.close(fig)
track_write(_p_conf)
print("saved:", _p_conf)

print()
print("Confidence summary statistics (fine-tuned model, max softmax prob per TEST sample):")
print(pd.Series(_max_conf).describe().to_string())

saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb05/confidence_distribution_test.png

Confidence summary statistics (fine-tuned model, max softmax prob per TEST sample):
count    3589.000000
mean        0.929233
std         0.133201
min         0.304773
25%         0.936312
50%         0.998112
75%         0.999995
max         1.000000


## 10. Export artifacts to `../outputs/`

- `nb05_test_metrics.json` &mdash; accuracy/macro-F1/weighted-F1 for both models, canonical and
  sensitivity-filtered, plus generalisation gaps and model size/param counts.
- `nb05_per_class_metrics.csv` &mdash; per-class precision/recall/F1/support, both models, `model`
  column.
- `nb05_confusion_matrix.csv` &mdash; long-form `(true_class, predicted_class, count, model)`.
- `nb05_test_probabilities_finetuned.csv` / `nb05_test_probabilities_frozen.csv` &mdash;
  per-sample softmax, unthresholded, one file per model (mirrors notebook 04's single-model
  convention).

In [13]:
written_before_10 = bucket_bytes(_WRITTEN_FILES)

# --- (a) nb05_test_metrics.json ----------------------------------------------
TEST_METRICS = {
    "n_test": int(len(test_df)),
    "class_names": CLASS_NAMES,
    "models": {
        _tag: {
            "model_file": os.path.basename(MODEL_PATHS[_tag]),
            "file_size_mb": MODEL_INFO[_tag]["file_size_mb"],
            "total_params": MODEL_INFO[_tag]["total_params"],
            "trainable_params": MODEL_INFO[_tag]["trainable_params"],
            "canonical": {
                "accuracy": HEADLINE[_tag]["accuracy"],
                "macro_f1": HEADLINE[_tag]["macro_f1"],
                "weighted_f1": HEADLINE[_tag]["weighted_f1"],
            },
            "sensitivity_filtered_excluding_quality_anomalies": {
                "n_excluded": SENSITIVITY[_tag]["n_excluded"],
                "n_kept": SENSITIVITY[_tag]["n_kept"],
                "accuracy": SENSITIVITY[_tag]["accuracy"],
                "macro_f1": SENSITIVITY[_tag]["macro_f1"],
            },
            "validation_baseline": {
                "source_run_id": VAL_BASELINE[_tag]["source_run_id"],
                "source_path": VAL_BASELINE[_tag]["source_path"],
                "val_accuracy": VAL_BASELINE[_tag]["val_accuracy"],
                "val_macro_f1": VAL_BASELINE[_tag]["val_macro_f1"],
            },
            "generalisation_gap": {
                "accuracy_gap_test_minus_val": GAPS[_tag]["accuracy_gap"],
                "macro_f1_gap_test_minus_val": GAPS[_tag]["macro_f1_gap"],
            },
        }
        for _tag in MODELS
    },
    "confidence_summary_finetuned": {
        "mean": float(np.mean(_max_conf)),
        "median": float(np.median(_max_conf)),
        "std": float(np.std(_max_conf)),
        "min": float(np.min(_max_conf)),
        "max": float(np.max(_max_conf)),
    },
    "top_off_diagonal_confusions": {
        _tag: [
            {"true_class": t, "predicted_class": p, "count": c, "pct_of_test": pct}
            for t, p, c, pct in top_off_diagonal(CM_RAW[_tag], CLASS_NAMES, k=5)
        ]
        for _tag in MODELS
    },
}
p_metrics = os.path.join(OUT_DIR, "nb05_test_metrics.json")
with open(p_metrics, "w", encoding="utf-8") as f:
    json.dump(TEST_METRICS, f, indent=2)
track_write(p_metrics)
print("saved:", p_metrics)

# --- (b) nb05_per_class_metrics.csv -------------------------------------------
_per_class_rows = []
for _tag in MODELS:
    for c in CLASS_NAMES:
        r = REPORTS[_tag][c]
        _per_class_rows.append({
            "model": _tag,
            "class": c,
            "label": CLASS_TO_IDX[c],
            "precision": float(r["precision"]),
            "recall": float(r["recall"]),
            "f1": float(r["f1-score"]),
            "support": int(r["support"]),
            "val_recall_baseline": VAL_BASELINE[_tag]["per_class"][c]["recall"],
            "val_f1_baseline": VAL_BASELINE[_tag]["per_class"][c]["f1"],
            "recall_delta_vs_val": float(r["recall"]) - VAL_BASELINE[_tag]["per_class"][c]["recall"],
        })
per_class_df = pd.DataFrame(_per_class_rows)
p_pc = os.path.join(OUT_DIR, "nb05_per_class_metrics.csv")
per_class_df.to_csv(p_pc, index=False)
track_write(p_pc)
print("saved:", p_pc, f"({len(per_class_df)} rows)")

# --- (c) nb05_confusion_matrix.csv (long-form) --------------------------------
_cm_rows = []
for _tag in MODELS:
    _cm = CM_RAW[_tag]
    for i, true_c in enumerate(CLASS_NAMES):
        for j, pred_c in enumerate(CLASS_NAMES):
            _cm_rows.append({
                "model": _tag, "true_class": true_c, "predicted_class": pred_c,
                "count": int(_cm[i, j]),
            })
cm_long_df = pd.DataFrame(_cm_rows)
p_cm = os.path.join(OUT_DIR, "nb05_confusion_matrix.csv")
cm_long_df.to_csv(p_cm, index=False)
track_write(p_cm)
print("saved:", p_cm, f"({len(cm_long_df)} rows)")

# --- (d) per-sample TEST probabilities, unthresholded, one file per model ----
_prob_paths = {}
for _tag in MODELS:
    _prob_df = pd.DataFrame({
        "basename": test_basenames,
        "true_label": y_test,
        "true_class": [CLASS_NAMES[i] for i in y_test],
        "predicted_label": PREDS[_tag],
        "predicted_class": [CLASS_NAMES[i] for i in PREDS[_tag]],
    })
    for i, c in enumerate(CLASS_NAMES):
        _prob_df[f"prob_{c}"] = PROBS[_tag][:, i]
    _prob_df["model"] = _tag

    _prob_cols = [f"prob_{c}" for c in CLASS_NAMES]
    _rowsums = _prob_df[_prob_cols].to_numpy().sum(axis=1)
    assert np.allclose(_rowsums, 1.0, atol=1e-3), f"'{_tag}' softmax rows do not sum to 1 on export"
    assert (_prob_df["basename"].str.startswith("PrivateTest_")).all(), \
        f"'{_tag}' probability export contains a non-PrivateTest_ sample"
    assert len(_prob_df) == n_test_measured, f"'{_tag}' probability export row count != TEST size"

    p_prob = os.path.join(OUT_DIR, f"nb05_test_probabilities_{_tag}.csv")
    _prob_df.to_csv(p_prob, index=False)
    track_write(p_prob)
    _prob_paths[_tag] = p_prob
    print("saved:", p_prob, f"({len(_prob_df)} rows - TEST only, NO threshold applied)")

written_after_10 = bucket_bytes(_WRITTEN_FILES)
print()
print(f"Artifact bytes written by section 10: {written_after_10 - written_before_10} "
      f"({(written_after_10 - written_before_10) / 1e6:.2f} MB)")

saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb05_test_metrics.json
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb05_per_class_metrics.csv (14 rows)
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb05_confusion_matrix.csv (98 rows)
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb05_test_probabilities_finetuned.csv (3589 rows - TEST only, NO threshold applied)
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb05_test_probabilities_frozen.csv (3589 rows - TEST only, NO threshold applied)

Artifact bytes written by section 10: 975508 (0.98 MB)


## 11. Run metadata &mdash; fill `RUN` and save

Final step: the `RUN` dict (redefined in section 0) is filled from the measured values above and
written via `save_run()`, exactly as notebooks 03/04 do.

In [14]:
RUN["model_version"] = [os.path.basename(MODEL_PATHS[_tag]) for _tag in MODELS]

RUN["hyperparameters"] = {
    "input_size": INPUT_SIZE,
    "channels": 3,
    "preprocessing": "grayscale replicated to 3 channels, bilinear resize to 96x96, "
                     "mobilenet_v2.preprocess_input",
    "batch_size": EVAL_BATCH_SIZE,
    "class_names": CLASS_NAMES,
    "models_evaluated": list(MODELS.keys()),
    "threshold_applied": False,
    "resampling": "none",
    "test_set_touch_count": 1,
    "quality_anomaly_exclusion_source": "image_flags.csv (is_anomaly == True), joined on file_path",
}

RUN["metrics"] = {
    "n_test": int(len(test_df)),
    "canonical": {_tag: HEADLINE[_tag] for _tag in MODELS},
    "sensitivity_filtered": {_tag: SENSITIVITY[_tag] for _tag in MODELS},
    "generalisation_gap": {_tag: GAPS[_tag] for _tag in MODELS},
    "validation_baseline": {
        _tag: {"source_run_id": VAL_BASELINE[_tag]["source_run_id"],
               "val_accuracy": VAL_BASELINE[_tag]["val_accuracy"],
               "val_macro_f1": VAL_BASELINE[_tag]["val_macro_f1"]}
        for _tag in MODELS
    },
    "model_info": {
        _tag: {"file_size_mb": MODEL_INFO[_tag]["file_size_mb"],
               "total_params": MODEL_INFO[_tag]["total_params"],
               "trainable_params": MODEL_INFO[_tag]["trainable_params"]}
        for _tag in MODELS
    },
    "confidence_summary_finetuned": TEST_METRICS["confidence_summary_finetuned"],
    "test_split_used": True,
}

_ft_gap = GAPS["finetuned"]["macro_f1_gap"]
_fz_gap = GAPS["frozen"]["macro_f1_gap"]
_ft_sens_delta = SENSITIVITY["finetuned"]["macro_f1"] - HEADLINE["finetuned"]["macro_f1"]
_fz_sens_delta = SENSITIVITY["frozen"]["macro_f1"] - HEADLINE["frozen"]["macro_f1"]

RUN["notes"] = (
    "Notebook 05 evaluates both saved models (fer_mobilenetv2_finetuned_96.keras from notebook "
    "04, and fer_mobilenetv2_frozen_96.keras from notebook 03) on the held-out FER-2013 TEST "
    "split (PrivateTest_, 3589 rows), touched exactly once, verified by a hard gate before any "
    "image was decoded. "
    f"Fine-tuned: TEST accuracy={HEADLINE['finetuned']['accuracy']:.4f}, "
    f"macro_f1={HEADLINE['finetuned']['macro_f1']:.4f}, versus its own VALIDATION baseline "
    f"macro_f1={VAL_BASELINE['finetuned']['val_macro_f1']:.4f} (run "
    f"{VAL_BASELINE['finetuned']['source_run_id']}) - generalisation gap "
    f"{_ft_gap:+.4f} macro-F1. "
    f"Frozen: TEST accuracy={HEADLINE['frozen']['accuracy']:.4f}, "
    f"macro_f1={HEADLINE['frozen']['macro_f1']:.4f}, versus its own VALIDATION baseline "
    f"macro_f1={VAL_BASELINE['frozen']['val_macro_f1']:.4f} (run "
    f"{VAL_BASELINE['frozen']['source_run_id']}) - generalisation gap {_fz_gap:+.4f} macro-F1. "
    f"Sensitivity analysis excluded {SENSITIVITY['finetuned']['n_excluded']} TEST images flagged "
    "is_anomaly==True in image_flags.csv (secondary table only; the canonical full-TEST numbers "
    "above are the headline result): fine-tuned macro-F1 changed by "
    f"{_ft_sens_delta:+.4f}, frozen macro-F1 changed by {_fz_sens_delta:+.4f} when those images "
    "were removed. No confidence threshold (tau_face_min) was applied anywhere in this notebook - "
    "that decision belongs to a later notebook. No resampling, no reweighting. CP-1 "
    "(calm/neutral confusion) is inspected via the top-5 off-diagonal confusion cells and the "
    "saved confusion-matrix heatmaps/CSV; FER-2013 has no 'calm' class, so this reads as "
    "neutral/sad/fear cross-confusion, visible in nb05_confusion_matrix.csv and the plots in "
    "plots/nb05/."
)

run_path = save_run(RUN)
track_write(run_path)

_written_total = bucket_bytes(_WRITTEN_FILES)
print()
print(f"Total artifact bytes written by this notebook: {_written_total} "
      f"({_written_total / 1e6:.2f} MB)")
print()
print("=" * 78)
print("NOTEBOOK 05 COMPLETE")
print("=" * 78)
print(f"Fine-tuned  TEST accuracy={HEADLINE['finetuned']['accuracy']:.4f}  "
      f"macro_f1={HEADLINE['finetuned']['macro_f1']:.4f}  (gap vs VAL: {_ft_gap:+.4f})")
print(f"Frozen      TEST accuracy={HEADLINE['frozen']['accuracy']:.4f}  "
      f"macro_f1={HEADLINE['frozen']['macro_f1']:.4f}  (gap vs VAL: {_fz_gap:+.4f})")

saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/run_20260828_052743.json

Total artifact bytes written by this notebook: 1373972 (1.37 MB)

NOTEBOOK 05 COMPLETE
Fine-tuned  TEST accuracy=0.6289  macro_f1=0.6036  (gap vs VAL: -0.0085)
Frozen      TEST accuracy=0.4795  macro_f1=0.4138  (gap vs VAL: -0.0265)
